In [1]:
import numpy as np
import pandas as pd
from pyhive import presto
from datetime import timedelta,datetime
import json
import requests
import math
import os


pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings("ignore")
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )
presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

In [3]:
rf_daily_df = pd.read_sql(f"""select * from hive.experiments_internal.rf_daily_30days_20250317""", conn2)

In [4]:
# olf_g2n_df = pd.read_sql(f"""select * from hive.experiments_internal.olf_g2n_30days_20250317""", conn3)
olf_g2n_df = pd.read_sql(f"""select * from hive.experiments_internal.olf_g2n_30days_20250317_v1""", conn2)

In [5]:
# rha_qc_segment_df = pd.read_sql(f"""select * from hive.experiments_internal.rha_qc_segment_20250406""", conn3)
rha_qc_segment_df = pd.read_sql(f"""select * from hive.experiments_internal.rha_qc_usecase_segment_20250406""", conn2)

In [6]:
olf_geo_usecase_df = pd.read_sql(f"""select * from hive.experiments_internal.olf_geo_usecase_20250317""", conn2)

In [7]:
merged_df = rha_qc_segment_df.merge(olf_g2n_df, on = 'customer_id', how='left')
merged_df = merged_df.merge(rf_daily_df, on = 'customer_id', how='left')
merged_df = merged_df.merge(olf_geo_usecase_df, on = 'customer_id', how='left')
merged_df.head()

,customer_id,segment,office_tag,os,travelridehailing,quick_commerce,rr_count,net_count,cobrm_count,cobra_count,ocara_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc,ocara_ttc,expired_count,aborted_count,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickylocation,net_psm_hotspot,ao_session,fe_session,rr_session,office_gross,transit_gross,office_net,transit_net
0,62e118454a9e687fb3ffa869,4-OTHER,other,Android,yes,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,NaN,NaN,NaN,NaN
1,65cab07e2b682d8cacc3511f,3-MONTHLY,office,Android,yes,yes,2,2,0,0,0,0,0,0,,,0,0,1,0,0,0,1,0,0,0,1,1,0,0,0,1,6.0,6.0,2.0,0,0,0,0
2,5d975d91941a7d1c51d4baab,2-WEEKLY,office,Android,yes,yes,54,43,0,6,5,11,3,1,70.16666666666667,227.0,0,0,37,8,0,8,37,0,0,3,6,29,0,0,3,4,45.0,38.0,40.0,0,11,0,6
3,67223f7b312bc65a84986978,3-MONTHLY,other,Android,yes,yes,3,3,0,0,0,0,0,0,,,0,0,2,0,0,0,3,0,0,0,0,3,0,0,0,0,3.0,3.0,3.0,0,0,0,0
4,62aeac6790c7986dbdb02877,3-MONTHLY,office,Android,yes,yes,21,8,0,5,7,12,0,0,113.2,487.7142857142857,1,0,6,0,0,0,3,0,0,0,2,1,0,0,0,0,16.0,14.0,14.0,6,0,2,0


In [8]:
merged_df.to_parquet('merged_df.parquet')

In [9]:
# rpc = net rides/net customers
# cobrm/cobra/ocara/quick_cobra/quick_ocara/(expired_count+aborted_count) % = (cobrm/cobra/ocara/quick_cobra/quick_ocara/(expired_count+aborted_count))/rr
# haps% = haps_orders/net_orders
# haps% = haps_orders/net_orders

# positive_bid% = positive_bid/bid_orders
# negative_bid% = negative_bid/bid_orders

#gross_psm_google_search% = gross_psm_google_search/rr
#net_psm_google_search% = net_psm_google_search/rr

#a02fe = fe_session/ao_session
#fe2rr = 
#ao2rr = 
#g2n = net_count/rr_count

#office_gross% = office_gross/rr
#transit_gross% = transit_gross/rr

#office_net% = office_net/rr
#transit_net% = transit_net/rr

In [26]:
merged_df = pd.read_parquet('merged_df.parquet')

In [27]:
merged_df.shape

(4929725, 39)

In [32]:
merged_df = merged_df[~merged_df['os'].isin(['Blank'])]

In [33]:
merged_df.customer_id.nunique()

4727191

In [34]:
merged_df.columns

Index(['customer_id', 'segment', 'office_tag', 'os', 'travelridehailing',
       'quick_commerce', 'rr_count', 'net_count', 'cobrm_count', 'cobra_count',
       'ocara_count', 'canceltn_count', 'quick_cobra', 'quick_ocara',
       'cobra_ttc', 'ocara_ttc', 'expired_count', 'aborted_count',
       'haps_orders', 'bid_orders', 'negative_bid', 'positive_bid',
       'gross_psm_gps', 'gross_psm_fav_recent_history',
       'gross_psm_google_search', 'gross_psm_stickylocation',
       'gross_psm_hotspot', 'net_psm_gps', 'net_psm_fav_recent_history',
       'net_psm_google_search', 'net_psm_fav_stickylocation',
       'net_psm_hotspot', 'ao_session', 'fe_session', 'rr_session',
       'office_gross', 'transit_gross', 'office_net', 'transit_net'],
      dtype='object')

In [35]:
merged_df.rename(columns={'net_psm_fav_stickylocation':'net_psm_stickylocation'}, inplace=True)

In [36]:
cols_to_convert = [  'rr_count', 'net_count', 'cobrm_count', 'cobra_count', 'ocara_count',
       'canceltn_count', 'quick_cobra', 'quick_ocara', 'cobra_ttc',
       'ocara_ttc', 'expired_count', 'aborted_count', 'haps_orders',
       'bid_orders', 'negative_bid', 'positive_bid', 'gross_psm_gps',
       'gross_psm_fav_recent_history', 'gross_psm_google_search', 'gross_psm_stickylocation', 'gross_psm_hotspot',
       'net_psm_gps', 'net_psm_fav_recent_history', 'net_psm_google_search', 'net_psm_stickylocation', 'net_psm_hotspot',
       'ao_session', 'fe_session', 'rr_session', 'office_gross',
       'transit_gross', 'office_net', 'transit_net']

for col in cols_to_convert:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

In [37]:
# cobra_ttc	ocara_ttc

In [38]:
merged_df['net_customers'] = merged_df['net_count'].apply(lambda x: 1 if x>0 else 0)
merged_df['gross_customers'] = merged_df['rr_count'].apply(lambda x: 1 if x>0 else 0)

In [39]:
def agg_func(col):
    
    grouper = col
    print(grouper)
    
    segment_agg_df = merged_df\
    .groupby(grouper)\
    .agg(
         gross_customers = ('gross_customers', 'sum'),
         net_customers = ('net_customers', 'sum'),
         
         ao_session = ('ao_session', 'sum'),
         fe_session = ('fe_session', 'sum'),
         rr_session = ('rr_session', 'sum'),

         rr_count = ('rr_count', 'sum'),
         net_count = ('net_count', 'sum'),
         cobrm_count = ('cobrm_count', 'sum'),
         cobra_count = ('cobra_count', 'sum'),
         ocara_count = ('ocara_count', 'sum'),
         expired_count = ('expired_count', 'sum'),
         aborted_count = ('aborted_count', 'sum'),
         canceltn_count = ('canceltn_count', 'sum'),
        
         quick_cobra = ('quick_cobra', 'sum'),
         quick_ocara = ('quick_ocara', 'sum'),
         
         cobra_ttc_mean = ('cobra_ttc', 'mean'),
         cobra_ttc_p50 = ('cobra_ttc', 'median'),
         cobra_ttc_p75=('cobra_ttc', lambda x: x.quantile(0.75)),
         cobra_ttc_p90=('cobra_ttc', lambda x: x.quantile(0.90)),
         cobra_ttc_p95=('cobra_ttc', lambda x: x.quantile(0.95)),
    
         ocara_ttc_mean = ('ocara_ttc', 'mean'),
         ocara_ttc_p50 = ('ocara_ttc', 'median'),
         ocara_ttc_p75=('ocara_ttc', lambda x: x.quantile(0.75)),
         ocara_ttc_p90=('ocara_ttc', lambda x: x.quantile(0.90)),
         ocara_ttc_p95=('ocara_ttc', lambda x: x.quantile(0.95)),
    
         haps_orders = ('haps_orders', 'sum'),
         bid_orders = ('bid_orders', 'sum'),
         negative_bid = ('negative_bid', 'sum'),
         positive_bid = ('positive_bid', 'sum'),
    
         gross_psm_gps = ('gross_psm_gps', 'sum'),
         gross_psm_fav_recent_history = ('gross_psm_fav_recent_history', 'sum'),
         gross_psm_google_search = ('gross_psm_google_search', 'sum'),
         gross_psm_stickylocation = ('gross_psm_stickylocation', 'sum'),
         gross_psm_hotspot = ('gross_psm_hotspot', 'sum'),
        
         net_psm_gps = ('net_psm_gps', 'sum'),
         net_psm_fav_recent_history = ('net_psm_fav_recent_history', 'sum'),
         net_psm_google_search = ('net_psm_google_search', 'sum'),
         net_psm_stickylocation = ('net_psm_stickylocation', 'sum'),
         net_psm_hotspot = ('net_psm_hotspot', 'sum'),
         
         office_gross = ('office_gross', 'sum'),
         transit_gross = ('transit_gross', 'sum'),
         office_net = ('office_net', 'sum'),
         transit_net = ('transit_net', 'sum'),
        )\
    .reset_index()\
    .sort_values(by=grouper, ascending=True)

    segment_agg_df['rpc'] = round(segment_agg_df['net_count']/segment_agg_df['net_customers'],2)

    segment_agg_df['a02fe'] = round(segment_agg_df['fe_session']*100/segment_agg_df['ao_session'],2)
    segment_agg_df['fe2rr'] = round(segment_agg_df['rr_session']*100/segment_agg_df['fe_session'],2)
    segment_agg_df['ao2rr'] = round(segment_agg_df['rr_session']*100/segment_agg_df['ao_session'],2)
    
    segment_agg_df['g2n'] = round(segment_agg_df['net_count']*100/segment_agg_df['rr_count'],2)
    segment_agg_df['haps%'] = round(segment_agg_df['haps_orders']*100/segment_agg_df['net_count'],2)

    # cobrm/cobra/ocara/quick_cobra/quick_ocara/(expired_count+aborted_count) % = (cobrm/cobra/ocara/quick_cobra/quick_ocara/(expired_count+aborted_count))/rr
    for col in ['cobrm_count','cobra_count','ocara_count']:
        segment_agg_df[col.split('_')[0]+'%'] = round(segment_agg_df[col]*100/segment_agg_df['rr_count'],2)
    
    for col in ['quick_cobra','quick_ocara']:
        segment_agg_df[col +'%'] = round(segment_agg_df[col]*100/segment_agg_df['rr_count'],2)
    
    segment_agg_df['expired%'] = round((segment_agg_df['expired_count'] + segment_agg_df['aborted_count'])*100/segment_agg_df['rr_count'],2)

    segment_agg_df['positive_bid%'] = round(segment_agg_df['positive_bid']*100/segment_agg_df['bid_orders'],2)
    segment_agg_df['negative_bid%'] = round(segment_agg_df['negative_bid']*100/segment_agg_df['bid_orders'],2)

    # 'gross_psm_gps','gross_psm_fav_recent_history', 'gross_psm_google_search', 'gross_psm_stickylocation', 'gross_psm_hotspot'
    for col in ['gross_psm_gps','gross_psm_fav_recent_history', 'gross_psm_google_search', 'gross_psm_stickylocation', 'gross_psm_hotspot']:
        segment_agg_df[col +'%'] = round(segment_agg_df[col]*100/segment_agg_df['rr_count'],2)
    # 'net_psm_gps','net_psm_fav_recent_history', 'net_psm_google_search', 'net_psm_fav_stickylocation', 'net_psm_hotspot'
    for col in ['net_psm_gps','net_psm_fav_recent_history', 'net_psm_google_search', 'net_psm_stickylocation', 'net_psm_hotspot']:
        segment_agg_df[col +'%'] = round(segment_agg_df[col]*100/segment_agg_df['net_count'],2)

    for col in ['office_gross','transit_gross']:
        segment_agg_df[col +'%'] = round(segment_agg_df[col]*100/segment_agg_df['rr_count'],2)

    for col in ['office_net','transit_net']:
        segment_agg_df[col +'%'] = round(segment_agg_df[col]*100/segment_agg_df['net_count'],2)

    return segment_agg_df

In [40]:
%%time
segment_agg_df = agg_func('segment')

segment
CPU times: user 2.32 s, sys: 472 ms, total: 2.79 s
Wall time: 2.83 s


In [41]:
# segment	os	travelridehailing	quick_commerce

In [42]:
%%time
segment_agg_df = agg_func(['segment'])

['segment']
CPU times: user 2.37 s, sys: 92.7 ms, total: 2.47 s
Wall time: 2.48 s


In [43]:
segment_agg_df

,segment,gross_customers,net_customers,ao_session,fe_session,rr_session,rr_count,net_count,cobrm_count,cobra_count,ocara_count,expired_count,aborted_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,rpc,a02fe,fe2rr,ao2rr,g2n,haps%,cobrm%,cobra%,ocara%,quick_cobra%,quick_ocara%,expired%,positive_bid%,negative_bid%,gross_psm_gps%,gross_psm_fav_recent_history%,gross_psm_google_search%,gross_psm_stickylocation%,gross_psm_hotspot%,net_psm_gps%,net_psm_fav_recent_history%,net_psm_google_search%,net_psm_stickylocation%,net_psm_hotspot%,office_gross%,transit_gross%,office_net%,transit_net%
0,1-DAILY,108794,108735,5518154.0,4132697.0,3664277.0,7185744.0,4390000.0,45581.0,1136316.0,1212460.0,398315.0,3037.0,2394357.0,170376.0,116458.0,119.734629,114.600000,148.916667,186.000000,214.600000,302.097945,259.714286,364.5,506.409586,626.500000,3480216.0,1836521.0,998236.0,838285.0,2491060.0,807033.0,553322.0,1124730.0,503604.0,1501823.0,503493.0,347600.0,689897.0,303567.0,1239133.0,302852.0,638465.0,168763.0,40.37,74.89,88.67,66.40,61.09,79.28,0.63,15.81,16.87,2.37,1.62,5.59,45.65,54.35,34.67,11.23,7.70,15.65,7.01,34.21,11.47,7.92,15.72,6.91,17.24,4.21,14.54,3.84
1,2-WEEKLY,296361,295982,6310806.0,4784936.0,3804192.0,7319640.0,4384895.0,54635.0,1199228.0,1233499.0,443511.0,3824.0,2487362.0,171810.0,103255.0,119.114419,110.571429,152.250000,200.333333,237.199091,328.508037,262.250000,402.0,602.000000,787.000000,3343613.0,1502824.0,764701.0,738123.0,2854741.0,621029.0,532462.0,1175038.0,529192.0,1694441.0,380817.0,325076.0,706976.0,312786.0,1316735.0,418296.0,656466.0,227661.0,14.81,75.82,79.50,60.28,59.91,76.25,0.75,16.38,16.85,2.35,1.41,6.11,49.12,50.88,39.00,8.48,7.27,16.05,7.23,38.64,8.68,7.41,16.12,7.13,17.99,5.71,14.97,5.19
2,3-MONTHLY,1955298,1826793,13963363.0,10902335.0,7219608.0,13860293.0,8438253.0,121771.0,2246016.0,2235856.0,807481.0,10808.0,4603643.0,292061.0,145875.0,118.866528,106.500000,156.500000,215.500000,257.000000,396.503502,282.000000,488.0,803.300000,1100.733333,6227068.0,2063309.0,849318.0,1213991.0,6465825.0,735854.0,826406.0,2087015.0,925269.0,3921724.0,454705.0,498862.0,1282868.0,556070.0,2151591.0,1092226.0,1130609.0,605715.0,4.62,78.08,66.22,51.70,60.88,73.80,0.88,16.20,16.13,2.11,1.05,5.90,58.84,41.16,46.65,5.31,5.96,15.06,6.68,46.48,5.39,5.91,15.20,6.59,15.52,7.88,13.40,7.18
3,4-OTHER,1016588,922236,5092674.0,3974463.0,2274186.0,3497380.0,2256447.0,34265.0,477536.0,520508.0,205097.0,3494.0,1032309.0,57513.0,28930.0,120.275495,106.333333,160.000000,224.000000,265.500000,471.819777,324.000000,585.0,986.000000,1367.325000,1613146.0,423092.0,145647.0,277445.0,1821826.0,100908.0,184206.0,511209.0,225328.0,1177883.0,62830.0,116002.0,332110.0,144876.0,505809.0,338669.0,294173.0,205081.0,2.45,78.04,57.22,44.66,64.52,71.49,0.98,13.65,14.88,1.64,0.83,5.96,65.58,34.42,52.09,2.89,5.27,14.62,6.44,52.20,2.78,5.14,14.72,6.42,14.46,9.68,13.04,9.09
4,5-INACTIVE,104390,95669,303452.0,271069.0,0.0,658628.0,411256.0,5174.0,106784.0,99830.0,35116.0,465.0,211788.0,13776.0,6754.0,119.095531,107.720779,157.000000,214.000000,254.666667,405.307243,286.500000,492.0,814.000000,1130.450000,308142.0,95962.0,47411.0,48551.0,304688.0,51145.0,45532.0,64281.0,41955.0,189657.0,31866.0,28258.0,40367.0,25820.0,93229.0,56385.0,51285.0,32234.0,4.30,89.33,0.00,0.00,62.44,74.93,0.79,16.21,15.16,2.09,1.03,5.40,50.59,49.41,46.26,7.77,6.91,9.76,6.37,46.12,7.75,6.87,9.82,6.28,14.16,8.56,12.47,7.84


In [34]:
segment_agg_df.to_clipboard(index=False)

In [35]:
segment_os = agg_func(['segment','os'])
segment_os

['segment', 'os']


,segment,os,gross_customers,net_customers,ao_session,fe_session,rr_session,rr_count,net_count,cobrm_count,cobra_count,ocara_count,expired_count,aborted_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,rpc,a02fe,fe2rr,ao2rr,g2n,haps%,cobrm%,cobra%,ocara%,quick_cobra%,quick_ocara%,expired%,positive_bid%,negative_bid%,gross_psm_gps%,gross_psm_fav_recent_history%,gross_psm_google_search%,gross_psm_stickylocation%,gross_psm_hotspot%,net_psm_gps%,net_psm_fav_recent_history%,net_psm_google_search%,net_psm_fav_stickylocation%,net_psm_hotspot%,office_gross%,transit_gross%,office_net%,transit_net%
0,1-DAILY,android,78864,78805,3615143.0,2929731.0,2589457.0,5162198.0,3168403.0,34609.0,851123.0,833753.0,271935.0,2345.0,1719485.0,124614.0,78987.0,118.677288,113.636364,147.111111,183.00,210.800000,310.477442,265.000000,374.307904,524.666667,653.582353,2516896.0,1233462.0,684562.0,548900.0,1920907.0,708457.0,455732.0,567192.0,350249.0,1161041.0,441357.0,287518.0,347490.0,211951.0,819756.0,222041.0,427143.0,123483.0,40.21,81.04,88.39,71.63,61.38,79.44,0.67,16.49,16.15,2.41,1.53,5.31,44.50,55.50,37.21,13.72,8.83,10.99,6.78,36.64,13.93,9.07,10.97,6.69,15.88,4.30,13.48,3.90
1,1-DAILY,iOS,29978,29978,1904141.0,1203822.0,1075549.0,2024697.0,1222268.0,10978.0,285378.0,378941.0,126435.0,692.0,675297.0,45795.0,37500.0,122.466054,117.111111,154.000000,193.48,223.850000,280.194240,248.075000,339.786957,456.660000,557.000000,963869.0,603293.0,313753.0,289540.0,570502.0,98694.0,97654.0,557701.0,153443.0,340983.0,62207.0,60131.0,342515.0,91661.0,419524.0,80867.0,211401.0,45295.0,40.77,63.22,89.34,56.48,60.37,78.86,0.54,14.09,18.72,2.26,1.85,6.28,47.99,52.01,28.18,4.87,4.82,27.54,7.58,27.90,5.09,4.92,28.02,7.50,20.72,3.99,17.30,3.71
2,2-WEEKLY,android,224366,224040,4372641.0,3552586.0,2797929.0,5454522.0,3287295.0,43112.0,917942.0,890897.0,312169.0,3069.0,1851951.0,128170.0,73175.0,117.978379,109.666667,150.375000,197.40,233.279167,338.310111,268.666667,414.666667,624.500000,818.000000,2515578.0,1058177.0,541513.0,516664.0,2309658.0,545980.0,443506.0,616512.0,385109.0,1379152.0,334422.0,271750.0,371157.0,228364.0,899125.0,318809.0,453668.0,173247.0,14.67,81.25,78.76,63.99,60.27,76.52,0.79,16.83,16.33,2.35,1.34,5.78,48.83,51.17,42.34,10.01,8.13,11.30,7.06,41.95,10.17,8.27,11.29,6.95,16.48,5.84,13.80,5.27
3,2-WEEKLY,iOS,72000,71947,1939687.0,1233605.0,1007172.0,1865202.0,1097641.0,11523.0,281302.0,342621.0,131350.0,755.0,635446.0,43640.0,30081.0,122.602868,113.500000,158.000000,209.00,248.000000,298.935098,245.250000,366.000000,535.000000,686.755357,828060.0,444668.0,223199.0,221469.0,545089.0,75062.0,88958.0,558564.0,144099.0,315292.0,46400.0,53328.0,335841.0,84429.0,417632.0,99490.0,202808.0,54416.0,15.26,63.60,81.64,51.92,58.85,75.44,0.62,15.08,18.37,2.34,1.61,7.08,49.81,50.19,29.22,4.02,4.77,29.95,7.73,28.72,4.23,4.86,30.60,7.69,22.39,5.33,18.48,4.96
4,3-MONTHLY,android,1623602,1514751,10711019.0,8773456.0,5678659.0,11115782.0,6796901.0,102816.0,1817876.0,1764490.0,624391.0,9218.0,3685182.0,231326.0,111433.0,118.343401,106.000000,155.571429,214.00,256.000000,410.190648,292.000000,506.000000,832.000000,1139.000000,5025191.0,1567936.0,645524.0,922412.0,5627630.0,654481.0,709699.0,1220445.0,726043.0,3427776.0,404451.0,429813.0,752373.0,437382.0,1587405.0,901057.0,846191.0,499494.0,4.49,81.91,64.73,53.02,61.15,73.93,0.92,16.35,15.87,2.08,1.00,5.70,58.83,41.17,50.63,5.89,6.38,10.98,6.53,50.43,5.95,6.32,11.07,6.44,14.28,8.11,12.45,7.35
5,3-MONTHLY,iOS,362254,337455,3256439.0,2133522.0,1543665.0,2846794.0,1700523.0,198

In [44]:
segment_os = agg_func(['segment','os'])
segment_os

['segment', 'os']


,segment,os,gross_customers,net_customers,ao_session,fe_session,rr_session,rr_count,net_count,cobrm_count,cobra_count,ocara_count,expired_count,aborted_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,rpc,a02fe,fe2rr,ao2rr,g2n,haps%,cobrm%,cobra%,ocara%,quick_cobra%,quick_ocara%,expired%,positive_bid%,negative_bid%,gross_psm_gps%,gross_psm_fav_recent_history%,gross_psm_google_search%,gross_psm_stickylocation%,gross_psm_hotspot%,net_psm_gps%,net_psm_fav_recent_history%,net_psm_google_search%,net_psm_stickylocation%,net_psm_hotspot%,office_gross%,transit_gross%,office_net%,transit_net%
0,1-DAILY,Android,74038,73980,3315004.0,2734023.0,2415978.0,4828690.0,2970560.0,32809.0,798231.0,773316.0,251530.0,2215.0,1604356.0,116150.0,73129.0,118.684802,113.611111,147.106203,183.000000,210.436364,312.476452,266.352941,377.272727,528.500,658.181818,2359684.0,1140787.0,632010.0,508777.0,1825066.0,693950.0,439559.0,474847.0,328774.0,1104250.0,432515.0,277721.0,291789.0,199490.0,755262.0,207685.0,395835.0,115289.0,40.15,82.47,88.37,72.88,61.52,79.44,0.68,16.53,16.02,2.41,1.51,5.25,44.60,55.40,37.80,14.37,9.10,9.83,6.81,37.17,14.56,9.35,9.82,6.72,15.64,4.30,13.33,3.88
1,1-DAILY,iOS,34756,34755,2203150.0,1398674.0,1248299.0,2357054.0,1419440.0,12772.0,338085.0,439144.0,146785.0,822.0,790001.0,54226.0,43329.0,121.938812,116.714286,153.000000,192.500000,222.500000,280.205694,247.600000,339.320513,457.125,558.333333,1120532.0,695734.0,366226.0,329508.0,665994.0,113083.0,113763.0,649883.0,174830.0,397573.0,70978.0,69879.0,398108.0,104077.0,483871.0,95167.0,242630.0,53474.0,40.84,63.49,89.25,56.66,60.22,78.94,0.54,14.34,18.63,2.30,1.84,6.26,47.36,52.64,28.26,4.80,4.83,27.57,7.42,28.01,5.00,4.92,28.05,7.33,20.53,4.04,17.09,3.77
2,2-WEEKLY,Android,212794,212474,4068013.0,3352361.0,2639827.0,5156335.0,3114796.0,41018.0,867253.0,837555.0,292728.0,2948.0,1745826.0,120177.0,68520.0,118.016054,109.750000,150.416667,197.500000,233.500000,340.747059,270.428571,418.000000,629.500,825.000000,2384578.0,988595.0,502050.0,486545.0,2223723.0,536255.0,429568.0,525104.0,364173.0,1330223.0,328561.0,263804.0,317067.0,216361.0,835558.0,301367.0,423576.0,163893.0,14.66,82.41,78.75,64.89,60.41,76.56,0.80,16.82,16.24,2.33,1.33,5.73,49.22,50.78,43.13,10.40,8.33,10.18,7.06,42.71,10.55,8.47,10.18,6.95,16.20,5.84,13.60,5.26
3,2-WEEKLY,iOS,83567,83508,2242793.0,1432575.0,1164365.0,2163305.0,1270099.0,13617.0,331975.0,395944.0,150783.0,876.0,741536.0,51633.0,34735.0,121.847580,112.866667,157.000000,207.000000,246.000000,298.387840,245.000000,365.444444,534.500,686.425000,959035.0,514229.0,262651.0,251578.0,631018.0,84774.0,102894.0,649934.0,165019.0,364218.0,52256.0,61272.0,389909.0,96425.0,481177.0,116929.0,232890.0,63768.0,15.21,63.87,81.28,51.92,58.71,75.51,0.63,15.35,18.30,2.39,1.61,7.01,48.92,51.08,29.17,3.92,4.76,30.04,7.63,28.68,4.11,4.82,30.70,7.59,22.24,5.41,18.34,5.02
4,3-MONTHLY,Android,1575811,1469487,10222027.0,8445132.0,5451598.0,10703240.0,6554320.0,99579.0,1746613.0,1694837.0,598830.0,8971.0,3541029.0,221214.0,106772.0,118.415497,106.000000,155.750000,214.000000,256.000000,412.738381,294.000000,509.000000,837.000,1146.000000,4846387.0,1493473.0,610269.0,883204.0,5507761.0,644306.0,693283.0,1082722.0,698213.0,3358392.0,398280.0,420405.0,669877.0,421123.0,1506816.0,869706.0,806163.0,482598.0,4.46,82.62,64.55,53.33,61.24,73.94,0.93,16.32,15.83,2.07,1.00,5.68,59.14,40.86,51.46,6.02,6.48,10.12,6.52,51.24,6.08,6.41,10.22,6.43,14.08,8.13,12.30,7.36
5,3-MONTHLY,iOS,379487,357306,3741336.0,2457203.0,1768010.0,3157053.0,1883933.0,2219

In [36]:
segment_os.to_clipboard(index=False)

In [37]:
segment_travel = agg_func(['segment','travelridehailing'])
segment_travel

['segment', 'travelridehailing']


,segment,travelridehailing,gross_customers,net_customers,ao_session,fe_session,rr_session,rr_count,net_count,cobrm_count,cobra_count,ocara_count,expired_count,aborted_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,rpc,a02fe,fe2rr,ao2rr,g2n,haps%,cobrm%,cobra%,ocara%,quick_cobra%,quick_ocara%,expired%,positive_bid%,negative_bid%,gross_psm_gps%,gross_psm_fav_recent_history%,gross_psm_google_search%,gross_psm_stickylocation%,gross_psm_hotspot%,net_psm_gps%,net_psm_fav_recent_history%,net_psm_google_search%,net_psm_fav_stickylocation%,net_psm_hotspot%,office_gross%,transit_gross%,office_net%,transit_net%
0,1-DAILY,iOS,29978,29978,1904141.0,1203822.0,1075549.0,2024697.0,1222268.0,10978.0,285378.0,378941.0,126435.0,692.0,675297.0,45795.0,37500.0,122.466054,117.111111,154.000000,193.480000,223.850000,280.194240,248.075000,339.786957,456.660000,557.000000,963869.0,603293.0,313753.0,289540.0,570502.0,98694.0,97654.0,557701.0,153443.0,340983.0,62207.0,60131.0,342515.0,91661.0,419524.0,80867.0,211401.0,45295.0,40.77,63.22,89.34,56.48,60.37,78.86,0.54,14.09,18.72,2.26,1.85,6.28,47.99,52.01,28.18,4.87,4.82,27.54,7.58,27.90,5.09,4.92,28.02,7.50,20.72,3.99,17.30,3.71
1,1-DAILY,no,10614,10594,486846.0,382979.0,336764.0,681495.0,435682.0,4249.0,111792.0,102585.0,26767.0,417.0,218626.0,17512.0,9958.0,110.258570,104.000000,137.000000,174.000000,203.000000,352.086695,296.875000,429.000000,612.886364,769.613333,343363.0,117081.0,62299.0,54782.0,317847.0,78004.0,56239.0,63945.0,30845.0,199607.0,51891.0,37123.0,41419.0,19577.0,74797.0,27322.0,43045.0,15869.0,41.13,78.67,87.93,69.17,63.93,78.81,0.62,16.40,15.05,2.57,1.46,3.99,46.79,53.21,46.64,11.45,8.25,9.38,4.53,45.81,11.91,8.52,9.51,4.49,10.98,4.01,9.88,3.64
2,1-DAILY,yes,68250,68211,3128297.0,2546752.0,2252693.0,4480703.0,2732721.0,30360.0,739331.0,731168.0,245168.0,1928.0,1500859.0,107102.0,69029.0,119.930693,115.090909,148.333333,184.166667,211.735714,304.073969,260.642857,366.307692,510.011765,631.216667,2173533.0,1116381.0,622263.0,494118.0,1603060.0,630453.0,399493.0,503247.0,319404.0,961434.0,389466.0,250395.0,306071.0,192374.0,744959.0,194719.0,384098.0,107614.0,40.06,81.41,88.45,72.01,60.99,79.54,0.68,16.50,16.32,2.39,1.54,5.51,44.26,55.74,35.78,14.07,8.92,11.23,7.13,35.18,14.25,9.16,11.20,7.04,16.63,4.35,14.06,3.94
3,2-WEEKLY,iOS,72000,71947,1939687.0,1233605.0,1007172.0,1865202.0,1097641.0,11523.0,281302.0,342621.0,131350.0,755.0,635446.0,43640.0,30081.0,122.602868,113.500000,158.000000,209.000000,248.000000,298.935098,245.250000,366.000000,535.000000,686.755357,828060.0,444668.0,223199.0,221469.0,545089.0,75062.0,88958.0,558564.0,144099.0,315292.0,46400.0,53328.0,335841.0,84429.0,417632.0,99490.0,202808.0,54416.0,15.26,63.60,81.64,51.92,58.85,75.44,0.62,15.08,18.37,2.34,1.61,7.08,49.81,50.19,29.22,4.02,4.77,29.95,7.73,28.72,4.23,4.86,30.60,7.69,22.39,5.33,18.48,4.96
4,2-WEEKLY,no,37105,37032,707983.0,558549.0,431802.0,855354.0,538627.0,6674.0,139120.0,134148.0,36117.0,662.0,279942.0,19609.0,10685.0,109.787731,100.000000,140.000000,186.633333,222.000000,388.952414,306.333333,479.500000,723.000000,945.000000,407934.0,119342.0,54672.0,64670.0,440827.0,71612.0,61629.0,82871.0,41239.0,273535.0,46832.0,39800.0,53269.0,25559.0,88296.0,50564.0,50591.0,28059.0,14.54,78.89,77.31,60.99,62.97,75.74,0.78,16.26,15.68,2.29,1.25,4.30,54.19,45.81,51.54,8.37,7.21,9.69,4.82,50.78,8.69,7.39,9.89,4.75,10.32,5.91,9.39,5.21
5,2-WEEKLY,yes,187261,187008,3664658.0,2994037.0,2366127.0,4599168.0,2748668.0,36438.0,778822.0,756749.0,276052.0,2407.0,1572009.0,108561.0,62

In [38]:
segment_travel.to_clipboard(index=False)

In [39]:
segment_qc = agg_func(['segment','quick_commerce'])
segment_qc

['segment', 'quick_commerce']


,segment,quick_commerce,gross_customers,net_customers,ao_session,fe_session,rr_session,rr_count,net_count,cobrm_count,cobra_count,ocara_count,expired_count,aborted_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,rpc,a02fe,fe2rr,ao2rr,g2n,haps%,cobrm%,cobra%,ocara%,quick_cobra%,quick_ocara%,expired%,positive_bid%,negative_bid%,gross_psm_gps%,gross_psm_fav_recent_history%,gross_psm_google_search%,gross_psm_stickylocation%,gross_psm_hotspot%,net_psm_gps%,net_psm_fav_recent_history%,net_psm_google_search%,net_psm_fav_stickylocation%,net_psm_hotspot%,office_gross%,transit_gross%,office_net%,transit_net%
0,1-DAILY,iOS,29978,29978,1904141.0,1203822.0,1075549.0,2024697.0,1222268.0,10978.0,285378.0,378941.0,126435.0,692.0,675297.0,45795.0,37500.0,122.466054,117.111111,154.000000,193.480000,223.850000,280.194240,248.075000,339.786957,456.660000,557.000000,963869.0,603293.0,313753.0,289540.0,570502.0,98694.0,97654.0,557701.0,153443.0,340983.0,62207.0,60131.0,342515.0,91661.0,419524.0,80867.0,211401.0,45295.0,40.77,63.22,89.34,56.48,60.37,78.86,0.54,14.09,18.72,2.26,1.85,6.28,47.99,52.01,28.18,4.87,4.82,27.54,7.58,27.90,5.09,4.92,28.02,7.50,20.72,3.99,17.30,3.71
1,1-DAILY,no,10481,10451,505901.0,396333.0,335109.0,674950.0,422581.0,5101.0,110939.0,102398.0,33508.0,420.0,218438.0,16388.0,10904.0,117.214619,111.777778,145.578947,182.295238,213.183333,360.209471,303.475000,435.566176,624.950000,791.325000,333504.0,115168.0,72186.0,42982.0,321288.0,80325.0,52768.0,61442.0,31895.0,197449.0,53001.0,34360.0,38516.0,20065.0,69120.0,28529.0,38223.0,15992.0,40.43,78.34,84.55,66.24,62.61,78.92,0.76,16.44,15.17,2.43,1.62,5.03,37.32,62.68,47.60,11.90,7.82,9.10,4.73,46.72,12.54,8.13,9.11,4.75,10.24,4.23,9.05,3.78
2,1-DAILY,yes,68383,68354,3109242.0,2533398.0,2254348.0,4487248.0,2745822.0,29508.0,740184.0,731355.0,238427.0,1925.0,1501047.0,108226.0,68083.0,118.894281,114.000000,147.300000,183.100000,210.500000,302.939016,259.533937,365.102778,508.366667,627.926786,2183392.0,1118294.0,612376.0,505918.0,1599619.0,628132.0,402964.0,505750.0,318354.0,963592.0,388356.0,253158.0,308974.0,191886.0,750636.0,193512.0,388920.0,107491.0,40.17,81.48,88.99,72.50,61.19,79.52,0.66,16.50,16.30,2.41,1.52,5.36,45.24,54.76,35.65,14.00,8.98,11.27,7.09,35.09,14.14,9.22,11.25,6.99,16.73,4.31,14.16,3.91
3,2-WEEKLY,iOS,72000,71947,1939687.0,1233605.0,1007172.0,1865202.0,1097641.0,11523.0,281302.0,342621.0,131350.0,755.0,635446.0,43640.0,30081.0,122.602868,113.500000,158.000000,209.000000,248.000000,298.935098,245.250000,366.000000,535.000000,686.755357,828060.0,444668.0,223199.0,221469.0,545089.0,75062.0,88958.0,558564.0,144099.0,315292.0,46400.0,53328.0,335841.0,84429.0,417632.0,99490.0,202808.0,54416.0,15.26,63.60,81.64,51.92,58.85,75.44,0.62,15.08,18.37,2.34,1.61,7.08,49.81,50.19,29.22,4.02,4.77,29.95,7.73,28.72,4.23,4.86,30.60,7.69,22.39,5.33,18.48,4.96
4,2-WEEKLY,no,33833,33693,678759.0,535575.0,396823.0,780598.0,480303.0,7299.0,126836.0,124761.0,40708.0,687.0,258896.0,17177.0,12130.0,114.713111,105.750000,146.333333,193.750000,231.000000,398.830901,311.500000,489.857143,749.185714,987.200000,364031.0,104489.0,54257.0,50232.0,414911.0,65785.0,55590.0,73991.0,37870.0,253594.0,42011.0,34876.0,46378.0,22615.0,79586.0,47499.0,43165.0,25413.0,14.26,78.91,74.09,58.46,61.53,75.79,0.94,16.25,15.98,2.20,1.55,5.30,48.07,51.93,53.15,8.43,7.12,9.48,4.85,52.80,8.75,7.26,9.66,4.71,10.20,6.08,8.99,5.29
5,2-WEEKLY,yes,190533,190347,3693882.0,3017011.0,2401106.0,4673924.0,2806992.0,35813.0,791106.0,766136.0,271461.0,2382.0,1593055.0,110993.0,6104

In [40]:
segment_qc.to_clipboard(index=False)

In [41]:
segment_travel_qc = agg_func(['segment','travelridehailing','quick_commerce'])
segment_travel_qc

['segment', 'travelridehailing', 'quick_commerce']


,segment,travelridehailing,quick_commerce,gross_customers,net_customers,ao_session,fe_session,rr_session,rr_count,net_count,cobrm_count,cobra_count,ocara_count,expired_count,aborted_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,rpc,a02fe,fe2rr,ao2rr,g2n,haps%,cobrm%,cobra%,ocara%,quick_cobra%,quick_ocara%,expired%,positive_bid%,negative_bid%,gross_psm_gps%,gross_psm_fav_recent_history%,gross_psm_google_search%,gross_psm_stickylocation%,gross_psm_hotspot%,net_psm_gps%,net_psm_fav_recent_history%,net_psm_google_search%,net_psm_fav_stickylocation%,net_psm_hotspot%,office_gross%,transit_gross%,office_net%,transit_net%
0,1-DAILY,iOS,iOS,29978,29978,1904141.0,1203822.0,1075549.0,2024697.0,1222268.0,10978.0,285378.0,378941.0,126435.0,692.0,675297.0,45795.0,37500.0,122.466054,117.111111,154.000000,193.480000,223.850000,280.194240,248.075000,339.786957,456.660000,557.000000,963869.0,603293.0,313753.0,289540.0,570502.0,98694.0,97654.0,557701.0,153443.0,340983.0,62207.0,60131.0,342515.0,91661.0,419524.0,80867.0,211401.0,45295.0,40.77,63.22,89.34,56.48,60.37,78.86,0.54,14.09,18.72,2.26,1.85,6.28,47.99,52.01,28.18,4.87,4.82,27.54,7.58,27.90,5.09,4.92,28.02,7.50,20.72,3.99,17.30,3.71
1,1-DAILY,no,no,4561,4543,214536.0,166390.0,143017.0,292184.0,184444.0,2034.0,48265.0,44356.0,12881.0,203.0,94655.0,7426.0,4628.0,113.229690,107.758373,139.983333,176.702381,209.000000,363.314949,309.275641,443.450000,633.000000,798.370000,145096.0,47688.0,28923.0,18765.0,145762.0,30770.0,21912.0,26965.0,12322.0,90336.0,20700.0,14329.0,17117.0,7797.0,29497.0,11334.0,16667.0,6546.0,40.60,77.56,85.95,66.66,63.13,78.67,0.70,16.52,15.18,2.54,1.58,4.48,39.35,60.65,49.89,10.53,7.50,9.23,4.22,48.98,11.22,7.77,9.28,4.23,10.10,3.88,9.04,3.55
2,1-DAILY,no,yes,6053,6051,272310.0,216589.0,193747.0,389311.0,251238.0,2215.0,63527.0,58229.0,13886.0,214.0,123971.0,10086.0,5330.0,108.023568,101.516667,135.000000,170.747222,199.408333,343.672631,289.187500,417.000000,594.066667,754.980000,198267.0,69393.0,33376.0,36017.0,172085.0,47234.0,34327.0,36980.0,18523.0,109271.0,31191.0,22794.0,24302.0,11780.0,45300.0,15988.0,26378.0,9323.0,41.52,79.54,89.45,71.15,64.53,78.92,0.57,16.32,14.96,2.59,1.37,3.62,51.90,48.10,44.20,12.13,8.82,9.50,4.76,43.49,12.41,9.07,9.67,4.69,11.64,4.11,10.50,3.71
3,1-DAILY,yes,no,5920,5908,291365.0,229943.0,192092.0,382766.0,238137.0,3067.0,62674.0,58042.0,20627.0,217.0,123783.0,8962.0,6276.0,120.228582,115.000000,149.000000,185.580000,215.250000,357.827923,300.309524,428.187500,620.162338,783.666667,188408.0,67480.0,43263.0,24217.0,175526.0,49555.0,30856.0,34477.0,19573.0,107113.0,32301.0,20031.0,21399.0,12268.0,39623.0,17195.0,21556.0,9446.0,40.31,78.92,83.54,65.93,62.21,79.12,0.80,16.37,15.16,2.34,1.64,5.45,35.89,64.11,45.86,12.95,8.06,9.01,5.11,44.98,13.56,8.41,8.99,5.15,10.35,4.49,9.05,3.97
4,1-DAILY,yes,yes,62330,62303,2836932.0,2316809.0,2060601.0,4097937.0,2494584.0,27293.0,676657.0,673126.0,224541.0,1711.0,1377076.0,98140.0,62753.0,119.903196,115.111111,148.266667,184.000000,211.333333,299.018584,257.105263,360.571429,500.000000,614.600000,1985125.0,1048901.0,579000.0,469901.0,1427534.0,580898.0,368637.0,468770.0,299831.0,854321.0,357165.0,230364.0,284672.0,180106.0,705336.0,177524.0,362542.0,98168.0,40.04,81.67,88.94,72.63,60.87,79.58,0.67,16.51,16.43,2.39,1.53,5.52,44.80,55.20,34.84,14.18,9.00,11.44,7.32,34.25,14.32,9.23,11.41,7.22,17.21,4.33,14.53,3.94
5,2-WEEKLY,iOS,iOS,72000,71947,1939687.0,1233605.0,1007172.0,1865202.0,1097641.0,11523.0,281302.0,342621.0,131350.0,755.0,635446.0,43640.0,30081.0,122

In [42]:
segment_travel_qc.to_clipboard(index=False)

In [43]:
segment_office = agg_func(['segment','office_tag'])
segment_office

['segment', 'office_tag']


,segment,office_tag,gross_customers,net_customers,ao_session,fe_session,rr_session,rr_count,net_count,cobrm_count,cobra_count,ocara_count,expired_count,aborted_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,rpc,a02fe,fe2rr,ao2rr,g2n,haps%,cobrm%,cobra%,ocara%,quick_cobra%,quick_ocara%,expired%,positive_bid%,negative_bid%,gross_psm_gps%,gross_psm_fav_recent_history%,gross_psm_google_search%,gross_psm_stickylocation%,gross_psm_hotspot%,net_psm_gps%,net_psm_fav_recent_history%,net_psm_google_search%,net_psm_fav_stickylocation%,net_psm_hotspot%,office_gross%,transit_gross%,office_net%,transit_net%
0,1-DAILY,office,87290,87281,4581081.0,3397011.0,3021321.0,5928348.0,3605494.0,37492.0,945498.0,1008376.0,329003.0,2459.0,1991366.0,141336.0,94912.0,120.381009,115.200000,149.327957,186.291429,215.0000,297.812437,258.055728,359.60000,495.653623,610.000000,2847104.0,1561452.0,823972.0,737480.0,2017622.0,616059.0,442159.0,973434.0,435825.0,1211982.0,380963.0,275718.0,595179.0,261483.0,1229936.0,252523.0,638544.0,141812.0,41.31,74.15,88.94,65.95,60.82,78.97,0.63,15.95,17.01,2.38,1.60,5.59,47.23,52.77,34.03,10.39,7.46,16.42,7.35,33.61,10.57,7.65,16.51,7.25,20.75,4.26,17.71,3.93
1,1-DAILY,other,21552,21502,938203.0,736542.0,643685.0,1258547.0,785177.0,8095.0,191003.0,204318.0,69367.0,578.0,403416.0,29073.0,21575.0,116.928261,111.750000,147.000000,185.000000,214.0000,319.614021,267.041667,385.65625,553.000000,695.186667,633661.0,275303.0,174343.0,100960.0,473787.0,191092.0,111227.0,151459.0,67867.0,290042.0,122601.0,71931.0,94826.0,42129.0,9344.0,50385.0,0.0,26966.0,36.52,78.51,87.39,68.61,62.39,80.70,0.64,15.18,16.23,2.31,1.71,5.56,36.67,63.33,37.65,15.18,8.84,12.03,5.39,36.94,15.61,9.16,12.08,5.37,0.74,4.00,0.00,3.43
2,2-WEEKLY,office,208986,208887,4705125.0,3529949.0,2833211.0,5452956.0,3251329.0,41189.0,905761.0,916773.0,335161.0,2714.0,1863723.0,128606.0,74181.0,120.366093,112.000000,153.500000,201.000000,237.5000,321.615561,259.833333,394.00000,584.750000,756.729167,2461682.0,1184047.0,586521.0,597526.0,2054099.0,423792.0,384803.0,930703.0,429884.0,1213577.0,257222.0,233589.0,557767.0,253720.0,1288970.0,313714.0,656476.0,171660.0,15.57,75.02,80.26,60.22,59.63,75.71,0.76,16.61,16.81,2.36,1.36,6.20,50.46,49.54,37.67,7.77,7.06,17.07,7.88,37.33,7.91,7.18,17.16,7.80,23.64,5.75,20.19,5.28
3,2-WEEKLY,other,87380,87100,1607203.0,1256242.0,971890.0,1866768.0,1133607.0,13446.0,293483.0,316745.0,108358.0,1110.0,623674.0,43204.0,29075.0,115.873228,107.000000,149.000000,198.333333,236.9625,345.616065,269.142857,425.00000,648.000000,864.466667,881956.0,318798.0,178191.0,140607.0,800648.0,197250.0,147661.0,244373.0,99324.0,480867.0,123600.0,91489.0,149231.0,59073.0,27787.0,104585.0,0.0,56003.0,13.02,78.16,77.36,60.47,60.73,77.80,0.72,15.72,16.97,2.31,1.56,5.86,44.11,55.89,42.89,10.57,7.91,13.09,5.32,42.42,10.90,8.07,13.16,5.21,1.49,5.60,0.00,4.94
4,3-MONTHLY,office,798335,773562,7216680.0,5545021.0,3873522.0,7468364.0,4567828.0,63664.0,1235223.0,1164363.0,432482.0,4760.0,2463250.0,160390.0,75716.0,120.447042,108.800000,157.166667,215.000000,256.0000,371.819832,273.250000,458.50000,737.000000,994.500000,3342729.0,1243636.0,513657.0,729979.0,3247154.0,375652.0,429706.0,1244477.0,580763.0,1978791.0,231683.0,261359.0,765694.0,353669.0,2000255.0,531977.0,1140121.0,295894.0,5.90,76.84,69.86,53.67,61.16,73.18,0.85,16.54,15.59,2.15,1.01,5.85,58.70,41.30,43.48,5.03,5.75,16.66,7.78,43.32,5.07,5.72,16.76,7.74,26.78,7.12,24.96,6.48
5,3-MONTHLY,other,1187521,1078644,6750778.0,5361957.0,3348802.0,6494212.0,3929596.0,59014

In [45]:
segment_office.to_clipboard(index=False)

In [46]:
segment_office_travel_qc = agg_func(['segment','office_tag','travelridehailing','quick_commerce'])
segment_office_travel_qc

['segment', 'office_tag', 'travelridehailing', 'quick_commerce']


,segment,office_tag,travelridehailing,quick_commerce,gross_customers,net_customers,ao_session,fe_session,rr_session,rr_count,net_count,cobrm_count,cobra_count,ocara_count,expired_count,aborted_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,rpc,a02fe,fe2rr,ao2rr,g2n,haps%,cobrm%,cobra%,ocara%,quick_cobra%,quick_ocara%,expired%,positive_bid%,negative_bid%,gross_psm_gps%,gross_psm_fav_recent_history%,gross_psm_google_search%,gross_psm_stickylocation%,gross_psm_hotspot%,net_psm_gps%,net_psm_fav_recent_history%,net_psm_google_search%,net_psm_fav_stickylocation%,net_psm_hotspot%,office_gross%,transit_gross%,office_net%,transit_net%
0,1-DAILY,office,iOS,iOS,26256,26256,1695370.0,1066830.0,955941.0,1805338.0,1087043.0,9941.0,256014.0,339373.0,112360.0,603.0,605328.0,40997.0,33720.0,123.195501,117.789474,154.391304,193.730909,225.000000,279.917513,248.650000,340.025000,455.000000,554.042857,854541.0,544663.0,275661.0,269002.0,505564.0,84394.0,85745.0,500650.0,139757.0,301060.0,53032.0,52400.0,307072.0,83352.0,417735.0,71817.0,211401.0,40380.0,41.40,62.93,89.61,56.39,60.21,78.61,0.55,14.18,18.80,2.27,1.87,6.26,49.39,50.61,28.00,4.67,4.75,27.73,7.74,27.70,4.88,4.82,28.25,7.67,23.14,3.98,19.45,3.71
1,1-DAILY,office,no,no,3087,3085,154177.0,118548.0,101751.0,207229.0,130806.0,1501.0,34628.0,31260.0,8895.0,138.0,67389.0,5130.0,2976.0,113.764107,109.000000,139.500000,174.533333,202.938462,357.728418,308.000000,434.281955,619.440278,772.303846,102095.0,34671.0,20738.0,13933.0,102341.0,19791.0,14967.0,20204.0,9633.0,63358.0,13189.0,9932.0,12813.0,6015.0,28979.0,8119.0,16667.0,4772.0,42.40,76.89,85.83,66.00,63.12,78.05,0.72,16.71,15.08,2.48,1.44,4.36,40.19,59.81,49.39,9.55,7.22,9.75,4.65,48.44,10.08,7.59,9.80,4.60,13.98,3.92,12.74,3.65
2,1-DAILY,office,no,yes,4302,4302,201380.0,159263.0,141958.0,287155.0,184149.0,1772.0,48056.0,43283.0,9740.0,153.0,93111.0,7483.0,3898.0,107.918005,101.300000,134.500000,170.500000,198.166667,336.456653,284.750000,409.000000,576.450000,716.455556,144409.0,53047.0,25105.0,27942.0,125848.0,32739.0,23999.0,28545.0,14939.0,79578.0,21322.0,15771.0,18611.0,9433.0,44853.0,12356.0,26378.0,7182.0,42.81,79.09,89.13,70.49,64.13,78.42,0.62,16.74,15.07,2.61,1.36,3.45,52.67,47.33,43.83,11.40,8.36,9.94,5.20,43.21,11.58,8.56,10.11,5.12,15.62,4.30,14.32,3.90
3,1-DAILY,office,yes,no,3955,3953,206586.0,161452.0,134413.0,266365.0,165875.0,2086.0,43467.0,40350.0,14424.0,161.0,85903.0,6028.0,3603.0,121.093737,115.909091,149.767857,185.180000,212.953846,355.838763,301.527273,422.812500,607.571429,754.891667,130500.0,47884.0,30595.0,17289.0,119398.0,32103.0,22021.0,25209.0,14590.0,73414.0,20694.0,14207.0,15555.0,9025.0,38399.0,12728.0,21556.0,7038.0,41.96,78.15,83.25,65.06,62.27,78.67,0.78,16.32,15.15,2.26,1.35,5.48,36.11,63.89,44.82,12.05,8.27,9.46,5.48,44.26,12.48,8.56,9.38,5.44,14.42,4.78,13.00,4.24
4,1-DAILY,office,yes,yes,49690,49685,2323568.0,1890918.0,1687258.0,3362261.0,2037621.0,22192.0,563333.0,554110.0,183584.0,1404.0,1139635.0,81698.0,50715.0,120.279269,115.437500,148.333333,184.000000,211.182857,295.747630,256.000000,356.632997,492.333333,602.847222,1615559.0,881187.0,471873.0,409314.0,1164471.0,447032.0,295427.0,398826.0,256906.0,694572.0,272726.0,183408.0,241128.0,153658.0,699970.0,147503.0,362542.0,82440.0,41.01,81.38,89.23,72.61,60.60,79.29,0.66,16.75,16.48,2.43,1.51,5.50,46.45,53.55,34.63,13.30,8.79,11.86,7.64,34.09,13.38,9.00,11.83,7.54,20.82,4.39,17.79,4.05
5,1-DAILY,other,iOS,iOS,3722,3722,208771.0,136992.0,119608.0,219359.0,135225.0,1037.0,29364.0,39568.0,14075.0,89.0,69

In [47]:
segment_office_travel_qc.to_clipboard(index=False)

In [31]:
segment_agg_df

,segment,gross_customers,net_customers,ao_session,fe_session,rr_session,rr_count,net_count,cobrm_count,cobra_count,ocara_count,expired_count,aborted_count,canceltn_count,quick_cobra,quick_ocara,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,haps_orders,bid_orders,negative_bid,positive_bid,gross_psm_gps,gross_psm_fav_recent_history,gross_psm_google_search,gross_psm_stickylocation,gross_psm_hotspot,net_psm_gps,net_psm_fav_recent_history,net_psm_google_search,net_psm_fav_stickylocation,net_psm_hotspot,office_gross,transit_gross,office_net,transit_net,rpc,a02fe,fe2rr,ao2rr,g2n,haps%,cobrm%,cobra%,ocara%,quick_cobra%,quick_ocara%,expired%,positive_bid%,negative_bid%,gross_psm_gps%,gross_psm_fav_recent_history%,gross_psm_google_search%,gross_psm_stickylocation%,gross_psm_hotspot%,net_psm_gps%,net_psm_fav_recent_history%,net_psm_google_search%,net_psm_fav_stickylocation%,net_psm_hotspot%,office_gross%,transit_gross%,office_net%,transit_net%
0,1-DAILY,108842,108783,5519284.0,4133553.0,3665006.0,7186895.0,4390671.0,45587.0,1136501.0,1212694.0,398370.0,3037.0,2394782.0,170409.0,116487.0,119.729164,114.600000,148.909091,186.000000,214.60125,302.081349,259.70,364.465199,506.408715,626.500,3480765.0,1836755.0,998315.0,838440.0,2491409.0,807151.0,553386.0,1124893.0,503692.0,1502024.0,503564.0,347649.0,690005.0,303612.0,1239280.0,302908.0,638544.0,168778.0,40.36,74.89,88.66,66.40,61.09,79.28,0.63,15.81,16.87,2.37,1.62,5.59,45.65,54.35,34.67,11.23,7.70,15.65,7.01,34.21,11.47,7.92,15.72,6.91,17.24,4.21,14.54,3.84
1,2-WEEKLY,296366,295987,6312328.0,4786191.0,3805101.0,7319724.0,4384936.0,54635.0,1199244.0,1233518.0,443519.0,3824.0,2487397.0,171810.0,103256.0,119.115607,110.571429,152.250000,200.333333,237.20000,328.506119,262.25,402.000000,602.000000,787.000,3343638.0,1502845.0,764712.0,738133.0,2854747.0,621042.0,532464.0,1175076.0,529208.0,1694444.0,380822.0,325078.0,706998.0,312793.0,1316757.0,418299.0,656476.0,227663.0,14.81,75.82,79.50,60.28,59.91,76.25,0.75,16.38,16.85,2.35,1.41,6.11,49.12,50.88,39.00,8.48,7.27,16.05,7.23,38.64,8.68,7.41,16.12,7.13,17.99,5.71,14.97,5.19
2,3-MONTHLY,1985856,1852206,13967458.0,10906978.0,7222324.0,13962576.0,8497424.0,122678.0,2263580.0,2253505.0,814412.0,10868.0,4639763.0,294305.0,147229.0,118.915389,106.500000,156.600000,216.000000,257.00000,396.200230,282.00,488.000000,803.000000,1100.000,6270154.0,2080555.0,854491.0,1226064.0,6498798.0,739479.0,831889.0,2106596.0,934696.0,3940569.0,456877.0,502006.0,1294603.0,561549.0,2170304.0,1100795.0,1140127.0,610300.0,4.59,78.09,66.22,51.71,60.86,73.79,0.88,16.21,16.14,2.11,1.05,5.91,58.93,41.07,46.54,5.30,5.96,15.09,6.69,46.37,5.38,5.91,15.24,6.61,15.54,7.88,13.42,7.18
3,4-OTHER,1047498,947804,5093346.0,3976431.0,2274994.0,3567488.0,2299745.0,35107.0,488247.0,531406.0,209405.0,3545.0,1054760.0,58969.0,29874.0,120.280797,106.200000,160.200000,224.000000,266.00000,470.181758,323.00,583.000000,983.000000,1364.000,1643422.0,432591.0,147732.0,284859.0,1844713.0,102617.0,187416.0,525516.0,231829.0,1191783.0,63822.0,117944.0,341453.0,148958.0,517771.0,345722.0,301006.0,209133.0,2.43,78.07,57.21,44.67,64.46,71.46,0.98,13.69,14.90,1.65,0.84,5.97,65.85,34.15,51.71,2.88,5.25,14.73,6.50,51.82,2.78,5.13,14.85,6.48,14.51,9.69,13.09,9.09
4,5-INACTIVE,107404,98363,303552.0,272022.0,0.0,675334.0,421041.0,5340.0,109779.0,102663.0,36031.0,477.0,217782.0,14253.0,7057.0,118.962526,107.666667,156.641026,214.000000,254.00000,403.746654,285.10,490.000000,811.450000,1126.475,315415.0,98786.0,48824.0,49962.0,309852.0,51648.0,45914.0,68714.0,42868.0,192505.0,32160.0,28489.0,43054.0,26364.0,96263.0,57758.0,52816.0,32976.0,4.28,89.61,0.00,0.00,62.35,74.91,0.79,16.26,15.20,2.11,1.04,5.41,50.58,49.42,45.88,7.65,6.80,10.17,6.35,45.72,7.64,6.77,10.23,6.26,14.25,8.55,12.54,7.83
